# Calibration of Example Smile

In [ ]:
using Pkg
Pkg.activate(".")

In [ ]:
using PiecewiseVanillaModel
using PlotlyJS

pvm = PiecewiseVanillaModel

In [ ]:
data = [
    # moneyness        Black vol (Case I)   Black vol (Case II)
    0.035123777453185  0.642412798191439    0.649712512502887
    0.049095433048156  0.621682849924325    0.629372247414191
    0.068624781300891  0.590577891369241    0.598339248024188
    0.095922580089594  0.553137221952525    0.560748840467284
    0.134078990076508  0.511398042127817    0.518685454812697
    0.18741338653678   0.466699250819768    0.473512707134552
    0.261963320525776  0.420225808661573    0.426434688827871
    0.366167980681693  0.373296313420122    0.378806875802102
    0.511823524787378  0.327557513727855    0.332366264644264
    0.715418426368358  0.285106482185545    0.289407658380454
    1                  0.249328882881654    0.253751752243855
    1.39778339939642   0.228967051575314    0.235378088110653
    1.95379843162821   0.220857187809035    0.235343538571543
    2.73098701349666   0.218762825294675    0.260395028879884
    3.81732831143284   0.218742183617652    0.31735041252779
    5.33579814376678   0.218432406892364    0.368205175099723
    7.45829006788743   0.217198426268117    0.417582432865276
    10.4250740447762   0.21573928902421     0.46323707706565
    14.5719954372667   0.214619929462215    0.504386489988866
    20.3684933182917   0.2141074555437      0.539752566560924
    28.4707418310251   0.21457985392644     0.566370957381163
]
T = 5.0722;
# display(M)

We specify some global plotting properties.

In [ ]:
font_size = 18;

We illustrate the input data.

In [ ]:
legend=attr(
    x=0.90,
    y=0.95,
    xanchor="right",
    yanchor="top",
    orientation="h",
)

layout = Layout(
    font = attr( size=font_size-2 ),
    title = attr(
        text = "Input log-normal implied volatilities",
        font = attr( size=font_size ),
    ),
    xaxis = attr( title_font_size=font_size ),
    yaxis = attr( title_font_size=font_size ),
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    legend = legend,
)

marker_1 = attr(
    symbol="square",
    line=attr(width=2, color="blue"),
    color="white", size=5,
)
marker_2 = attr(
    symbol="square",
    line=attr(width=2, color="red"),
    color="white", size=5,
)

fig = plot(
    [
        scatter(x=log.(data[:,1]), y=data[:,2] * 1.e2, mode="markers", name="Case I", marker=marker_1),
        scatter(x=log.(data[:,1]), y=data[:,3] * 1.e2, mode="markers", name="Case II", marker=marker_2),
    ],
    layout
)

We calculate the data-implied probability densities.

In [ ]:
function data_implied_densities(strikes, volatilities, s0, T)
    dens = []
    for k = 2:length(strikes)-1
        if strikes[k] ≥ s0
            cp = 1.0
        else
            cp = -1.0
        end
        p0 = pvm.black_price(strikes[k-1], s0, volatilities[k-1], T, cp)
        p1 = pvm.black_price(strikes[k],   s0, volatilities[k],   T, cp)
        p2 = pvm.black_price(strikes[k+1], s0, volatilities[k+1], T, cp)
        d0 = (p1 - p0) / (strikes[k] - strikes[k-1])
        d1 = (p2 - p1) / (strikes[k+1] - strikes[k])
        q  = (d1 - d0) / (strikes[k+1] - strikes[k-1]) / 2.0
        dens = vcat(dens, q)
    end
    return (strikes[2:length(strikes)-1], dens)
end

data_dens_1 = data_implied_densities(data[:,1], data[:,2], 1.0, T)
data_dens_2 = data_implied_densities(data[:,1], data[:,3], 1.0, T)
;

In [ ]:
legend=attr(
    x=0.98,
    y=0.95,
    xanchor="right",
    yanchor="top",
    orientation="h",
)

layout = Layout(
    font = attr( size=font_size-2 ),
    title = attr(
        text = "Input data-implied densities (log-scale)",
        font = attr( size=font_size ),
    ),
    xaxis = attr( title_font_size=font_size ),
    yaxis = attr( title_font_size=font_size ),
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Density",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    legend = legend,
    yaxis_type="log"
)

marker_1 = attr(
    symbol="square",
    line=attr(width=2, color="blue"),
    color="white", size=5,
)
marker_2 = attr(
    symbol="square",
    line=attr(width=2, color="red"),
    color="white", size=5,
)

fig = plot(
    [
        scatter(x=log.(data_dens_1[1]), y=data_dens_1[2], mode="markers", name="Case I", marker=marker_1),
        scatter(x=log.(data_dens_2[1]), y=data_dens_2[2], mode="markers", name="Case II", marker=marker_2),
    ],
    layout
)

Our calibration methodology is based on normal volatilities. So, we translate log-normal volatilities to normal volatilities.

In [ ]:
function normal_volatility(black_vol, strike, forward, T)
    if strike ≥ forward
        cp = 1
    else
        cp = -1
    end
    price = pvm.black_price(strike, forward, black_vol, T, cp)
    n_vol = pvm.bachelier_implied_volatility(price, strike, forward, T, cp)
    return n_vol
end

In [ ]:
function black_volatility(m, strike)
    if strike ≥ m.s0
        cp = 1.0
        o = pvm.call_option(m, strike)
    else
        cp = -1.0
        o = pvm.put_option(m, strike)
    end
    v = pvm.black_implied_volatility(o, strike, m.s0, m.T, cp)
    return v
end

We calculate and plot the normal volatilities for our test case.

In [ ]:
n_vols_1 = [
    normal_volatility(black_vol, strike, 1.0, T)
    for (strike, black_vol) in zip(data[:,1], data[:,2])
]
n_vols_2 = [
    normal_volatility(black_vol, strike, 1.0, T)
    for (strike, black_vol) in zip(data[:,1], data[:,3])
]

layout = Layout(
    title="Input normal implied volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
)
marker_1 = attr(
    symbol="square",
    line=attr(width=2, color="blue"),
    color="white", size=5,
)
marker_2 = attr(
    symbol="square",
    line=attr(width=2, color="red"),
    color="white", size=5,
)
#
plot(
    [
        scatter(x=log.(data[:,1]), y=n_vols_1 * 1.e+2, mode="markers", name="Case I", marker = marker_1),
        scatter(x=log.(data[:,1]), y=n_vols_2 * 1.e+2, mode="markers", name="Case II", marker = marker_2),
    ],
    layout,
)

## Example Case I

In [ ]:
input_strikes    = data[:,1]
input_black_vols = data[:,2]
input_vols       = n_vols_1
;

We prepare data for model calibration.

In [ ]:
atm_strike = input_strikes[11]
atm_vol    = input_vols[11]
#
rel_strikes = input_strikes .- atm_strike
rel_strikes = vcat(rel_strikes[1:10], rel_strikes[12:21])
normal_vols = vcat(input_vols[1:10], input_vols[12:21])
#
dsl = reverse(-rel_strikes[1:10])
dsu = rel_strikes[11:20]
#
α = 0.0
lmfit_kwargs = (
    autodiff = :forwarddiff,
    maxIter  = 100,
    show_trace = false,
)
#
(m, res) = pvm.calibrated_model_from_smile(
    atm_strike, atm_vol, T, dsl, dsu,
    rel_strikes, normal_vols;
    rexl = nothing, rexu = 0.2,
    α=α, lmfit_kwargs = lmfit_kwargs,
)
#
vol_model_implied = [
    pvm.normal_volatility(m, strike)
    for strike in m.s0 .+ rel_strikes
]
fit = (vol_model_implied .- normal_vols) .* 1e+4  # in bp
display(maximum(abs.(fit)))
println("dvl:")
println(m.dvl)
println("dvu:")
println(m.dvu)
println("Fit:")
println(fit)
println("Converged: " * string(res.converged))

We plot calibrated local and implied normal volatilities.

In [ ]:
local_vols = [
    pvm.local_volatility(m, s) for s in input_strikes
]
normal_vols = [
    pvm.normal_volatility(m, s) for s in input_strikes
]

layout = Layout(
    title="Calibrated normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
)
#
plot(
    [
        scatter(x=log.(input_strikes), y=local_vols * 1.e+2, mode="markers", name="Local volatilities"),
        scatter(x=log.(input_strikes), y=input_vols * 1.e+2, mode="markers", name="Input normal volatilities"),
        scatter(x=log.(input_strikes), y=normal_vols * 1.e+2, mode="markers", name="Model normal volatilities"),
    ],
    layout
)

In [ ]:
model_strikes = input_strikes[begin]:0.01:input_strikes[end]

black_vols = [
    black_volatility(m, s) for s in model_strikes
]
#
legend=attr(
    x=0.90,
    y=0.95,
    xanchor="right",
    yanchor="top",
    orientation="v",
)
#
layout = Layout(
    title="Example Case I - Calibrated log-normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    legend = legend
)
#
marker_1 = attr(
    symbol="square",
    line=attr(width=2, color="blue"),
    color="white", size=5,
)
line_2 = attr(
    width=2,
    color="blue",
)
#

fig = plot(
    [
        scatter(x=log.(input_strikes), y=input_black_vols * 1.e+2, mode="markers", name="Input log-normal volatilities", marker = marker_1),
        scatter(x=log.(model_strikes), y=black_vols * 1.e+2, mode="lines", name="Model log-normal volatilities", line = line_2),
    ],
    layout
)

In [ ]:
black_vols = [
    black_volatility(m, s) for s in input_strikes
]

layout = Layout(
    title="Example Case I - Calibration fit (model vs. input) log-normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility difference (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    yaxis_range=[-1.0, 1.0] .* 1.0e-3,
)
#
marker_1 = attr(
    symbol="circle",
    line=attr(width=0, color="blue"),
    color="blue", size=5,
)
fig = plot(
    [
        scatter(x=log.(input_strikes), y=(black_vols - input_black_vols)*1.e+2, mode="markers", name="Black vols diff", marker = marker_1),
    ],
    layout
)

Similarly, we also plot implied the model-implied density.

In [ ]:
density_strikes = exp(-3.0):0.01:exp(3.0)
densities = [
    pvm.implied_density(m, s) for s in density_strikes
]

layout = Layout(
    title="Example Case I - Model-implied probability density",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Density",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    xaxis_range=[-3.0, 3.0],
)
line_1 = attr(
    width=2,
    color="blue",
)
#
fig = plot(
    [
        scatter(x=log.(density_strikes), y=densities, mode="lines", name="Density", line = line_1),
    ],
    layout
)

In [ ]:
layout = Layout(
    title="Example Case I - Model-implied probability density (log-scale)",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Density",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    xaxis_range=[-3.0, 3.0],
    yaxis_type="log"
)
line_1 = attr(
    width=2,
    color="blue",
)
#
fig = plot(
    [
        scatter(x=log.(density_strikes), y=densities, mode="lines", name="Density", line = line_1),
    ],
    layout
)

## Example Case II

In [ ]:
input_strikes    = data[:,1]
input_black_vols = data[:,3]
input_vols       = n_vols_2
;

In [ ]:
atm_strike = input_strikes[11]
atm_vol    = input_vols[11]
#
rel_strikes = input_strikes .- atm_strike
rel_strikes = vcat(rel_strikes[1:10], rel_strikes[12:21])
normal_vols = vcat(input_vols[1:10], input_vols[12:21])
#
dsl = reverse(-rel_strikes[1:10])
dsu = rel_strikes[11:20]
#
α = 0.0
lmfit_kwargs = (
    autodiff = :forwarddiff,
    maxIter  = 100,
    show_trace = false,
)
#
(m, res) = pvm.calibrated_model_from_smile(
    atm_strike, atm_vol, T, dsl, dsu,
    rel_strikes, normal_vols;
    rexl = nothing, rexu = 0.0,
    α=α, lmfit_kwargs = lmfit_kwargs,
)
#
vol_model_implied = [
    pvm.normal_volatility(m, strike)
    for strike in m.s0 .+ rel_strikes
]
fit = (vol_model_implied .- normal_vols) .* 1e+4  # in bp
display(maximum(abs.(fit)))
println("dvl:")
println(m.dvl)
println("dvu:")
println(m.dvu)
println("Fit:")
println(fit)
println("Converged: " * string(res.converged))

In [ ]:
local_vols = [
    pvm.local_volatility(m, s) for s in input_strikes
]
normal_vols = [
    pvm.normal_volatility(m, s) for s in input_strikes
]

layout = Layout(
    title="Calibrated normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
)
#
plot(
    [
        scatter(x=log.(input_strikes), y=local_vols * 1.e+2, mode="markers", name="Local volatilities"),
        scatter(x=log.(input_strikes), y=input_vols * 1.e+2, mode="markers", name="Input normal volatilities"),
        scatter(x=log.(input_strikes), y=normal_vols * 1.e+2, mode="markers", name="Model normal volatilities"),
    ],
    layout
)

In [ ]:
model_strikes = input_strikes[begin]:0.01:input_strikes[end]
black_vols = [
    black_volatility(m, s) for s in model_strikes
]
#
legend=attr(
    x=0.90,
    y=0.95,
    xanchor="right",
    yanchor="top",
    orientation="v",
)
#
layout = Layout(
    title="Example Case II - Calibrated log-normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    legend = legend
)
#
marker_1 = attr(
    symbol="square",
    line=attr(width=2, color="red"),
    color="white", size=5,
)
line_2 = attr(
    width=2,
    color="red",
)
#

fig = plot(
    [
        scatter(x=log.(input_strikes), y=input_black_vols * 1.e+2, mode="markers", name="Input log-normal volatilities", marker = marker_1),
        scatter(x=log.(model_strikes), y=black_vols * 1.e+2, mode="lines", name="Model log-normal volatilities", line = line_2),
    ],
    layout
)

In [ ]:
black_vols = [
    black_volatility(m, s) for s in input_strikes
]

layout = Layout(
    title="Example Case II - Calibration fit (model vs. input) log-normal volatilities",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Volatility difference (%)",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    yaxis_range=[-1.0, 1.0] .* 1.0,
)
#
marker_1 = attr(
    symbol="circle",
    line=attr(width=0, color="red"),
    color="red", size=5,
)
fig = plot(
    [
        scatter(x=log.(input_strikes), y=(black_vols - input_black_vols)*1.e+2, mode="markers", name="Black vols diff", marker = marker_1),
    ],
    layout
)

In [ ]:
density_strikes = exp(-3.0):0.01:exp(3.0)
densities = [
    pvm.implied_density(m, s) for s in density_strikes
]

layout = Layout(
    title="Example Case II - Model-implied probability density",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Density",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    xaxis_range=[-3.0, 3.0],
)
line_1 = attr(
    width=2,
    color="red",
)
#
fig = plot(
    [
        scatter(x=log.(density_strikes), y=densities, mode="lines", name="Density", line = line_1),
    ],
    layout
)

In [ ]:
layout = Layout(
    title="Example Case II - Model-implied probability density (log-scale)",
    xaxis_title="Log-moneyness, log(strike / forward)",
    yaxis_title="Density",
    legend_title="Legend title",
    autosize=false,
    width=800,
    height=600,
    xaxis_range=[-3.0, 3.0],
    yaxis_type="log"
)
line_1 = attr(
    width=2,
    color="red",
)
#
fig = plot(
    [
        scatter(x=log.(density_strikes), y=densities, mode="lines", name="Density", line = line_1),
    ],
    layout
)